# Download and refresh local price CSV files

Uses `download_adj_close_prices` from `data_loader.py` with provider fallback chain: `yfinance` -> `defeatbeta_api`.

It downloads each ticker in a specified date range and saves one CSV per ticker in `data/` with the same format used by the project (`Date,Adj Close`).

In [14]:
from __future__ import annotations

from pathlib import Path
import importlib
import time
import pandas as pd

import data_loader as _data_loader
importlib.reload(_data_loader)
download_adj_close_prices = _data_loader.download_adj_close_prices

In [15]:
TICKERS = [
    "CSSPX.MI",
    "SWDA.MI",
    "EIMI.MI",
    "CSNDX.MI",
    "IUSQ.MI",
    "VWCE.MI",
    "EXS1.MI",
    "MEUD.MI",
    "SMEA.MI",
    "XD9U.MI",
    "XMME.MI",
    "CSSX5E.MI",
    "EMUL.MI",
    "XLRE",
    "GOVT",
    "TLH",
    "LTPZ",
    "XTLT.TO",
    "UTHY",
    "TIP",
    "IGLA.L",
    "XG7S.DE",
    "SEGA.MI",
    "VAGF.DE",
    "BTC=F",
    "GC=F",
    "SI=F",
    "CL=F",
    "CSH.PA",
    "PDBC",
    "VGLT",
]

In [16]:
# Configure your download window and tickers here
# Full ticker universe from the benchmark table (duplicates removed)

START_DATE = "2015-04-03"
END_DATE = "2026-04-14"

# Small delay between requests helps reduce provider throttling
COOLDOWN_SECONDS = 4

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Tickers: {TICKERS}")
print(f"Window: {START_DATE} -> {END_DATE}")
print(f"Output dir: {DATA_DIR.resolve()}")

Tickers: ['CSSPX.MI', 'SWDA.MI', 'EIMI.MI', 'CSNDX.MI', 'IUSQ.MI', 'VWCE.MI', 'EXS1.MI', 'MEUD.MI', 'SMEA.MI', 'XD9U.MI', 'XMME.MI', 'CSSX5E.MI', 'EMUL.MI', 'XLRE', 'GOVT', 'TLH', 'LTPZ', 'XTLT.TO', 'UTHY', 'TIP', 'IGLA.L', 'XG7S.DE', 'SEGA.MI', 'VAGF.DE', 'BTC=F', 'GC=F', 'SI=F', 'CL=F', 'CSH.PA', 'PDBC', 'VGLT']
Window: 2015-04-03 -> 2026-04-14
Output dir: /Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data


In [17]:
results = []

for i, ticker in enumerate(TICKERS, start=1):
    print(f"[{i}/{len(TICKERS)}] Downloading {ticker}...")
    status = "ok"
    rows = 0
    error = ""

    out_path = DATA_DIR / f"{ticker}.csv"

    try:
        req_start = pd.Timestamp(START_DATE)
        req_end = pd.Timestamp(END_DATE)
        one_day = pd.Timedelta(days=1)

        existing_series = None
        local_min = None
        local_max = None

        if out_path.exists():
            existing_df = pd.read_csv(out_path, parse_dates=["Date"])
            if not existing_df.empty and {"Date", "Adj Close"}.issubset(existing_df.columns):
                existing_series = (
                    existing_df.set_index("Date")["Adj Close"]
                    .astype(float)
                    .dropna()
                    .sort_index()
                )
                if not existing_series.empty:
                    local_min = existing_series.index.min()
                    local_max = existing_series.index.max()

        needs_left = local_min is None or req_start < local_min
        needs_right = local_max is None or req_end > local_max

        if not needs_left and not needs_right:
            # Requested window is fully covered locally: keep file untouched.
            final_series = existing_series
            print("    local data already covers requested window; no download needed")
        else:
            new_parts = []

            if needs_left:
                left_end = (local_min - one_day) if local_min is not None else req_end
                if req_start <= left_end:
                    left_prices = download_adj_close_prices(
                        tickers=[ticker],
                        start=req_start.strftime("%Y-%m-%d"),
                        end=left_end.strftime("%Y-%m-%d"),
                    )
                    left_series = left_prices[ticker].dropna()
                    new_parts.append(left_series)

            if needs_right:
                right_start = (local_max + one_day) if local_max is not None else req_start
                if right_start <= req_end:
                    right_prices = download_adj_close_prices(
                        tickers=[ticker],
                        start=right_start.strftime("%Y-%m-%d"),
                        end=req_end.strftime("%Y-%m-%d"),
                    )
                    right_series = right_prices[ticker].dropna()
                    new_parts.append(right_series)

            pieces = []
            if existing_series is not None and not existing_series.empty:
                pieces.append(existing_series)
            pieces.extend(new_parts)

            if not pieces:
                raise RuntimeError("No local data and no downloaded data available.")

            final_series = pd.concat(pieces).sort_index()
            final_series = final_series[~final_series.index.duplicated(keep="last")]

        # Force project-consistent on-disk shape: Date index + Adj Close column.
        out_df = final_series.to_frame(name="Adj Close")
        out_df.index.name = "Date"
        out_df.to_csv(out_path, date_format="%Y-%m-%d")

        rows = int(out_df.shape[0])
        print(f"    saved {rows} rows to {out_path}")
    except Exception as exc:
        status = "failed"
        error = str(exc)
        print(f"    FAILED: {ticker}: {error}")

    results.append({"ticker": ticker, "status": status, "rows": rows, "error": error})

    if i < len(TICKERS):
        time.sleep(COOLDOWN_SECONDS)

summary = pd.DataFrame(results)
summary

[1/31] Downloading CSSPX.MI...
    local data already covers requested window; no download needed
    saved 4038 rows to data/CSSPX.MI.csv
[2/31] Downloading SWDA.MI...
    local data already covers requested window; no download needed
    saved 4069 rows to data/SWDA.MI.csv
[3/31] Downloading EIMI.MI...
    local data already covers requested window; no download needed
    saved 2935 rows to data/EIMI.MI.csv
[4/31] Downloading CSNDX.MI...
    local data already covers requested window; no download needed
    saved 4063 rows to data/CSNDX.MI.csv


/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:322: RuntimeWarning: Local cache miss for symbols: ['IUSQ.MI']. Attempting online providers.
  f"Local cache miss for symbols: {missing_only}. Attempting online providers.",

1 Failed download:
['IUSQ.MI']: YFTzMissingError('possibly delisted; no timezone found')


[5/31] Downloading IUSQ.MI...
    FAILED: IUSQ.MI: No price data returned by yfinance for requested inputs.


/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:375: RuntimeWarning: Batch yfinance download returned incomplete data. Falling back to per-ticker recovery for: ['IUSQ.MI']
  (
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:384: RuntimeWarning: Yahoo Finance batch response returned no usable rows for all uncached symbols. Likely temporary provider throttling; skipping repeated per-ticker Yahoo calls.
  (
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:177: RuntimeWarning: defeatbeta_api is not installed/importable, so fallback data cannot be fetched.
  "defeatbeta_api is not installed/importable, so fallback data cannot be fetched.",
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:405: RuntimeWarning: defeatbeta_api returned no usable rows for IUSQ.MI.
  f"defeatbeta_api returned no usable rows for {symbol}.",
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:410: RuntimeWarning:

[6/31] Downloading VWCE.MI...


...

/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:322: RuntimeWarning: Local cache miss for symbols: ['BTC=F']. Attempting online providers.
  f"Local cache miss for symbols: {missing_only}. Attempting online providers.",
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/yfinance/scrapers/history.py:169: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:375: RuntimeWarning: Batch yfinance download returned incomplete data. Falling back to per-ticker recovery for: ['BTC=F']
  (
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:384: RuntimeWarning: Yahoo Finance batch response returned no usable rows for all uncached symbols. Likely temporary provider throttling; skipping repeated per-ticker Yahoo calls.
  (
/Users/giacomomaggiore/Desktop/CODING/l

[26/31] Downloading GC=F...
    local data already covers requested window; no download needed
    saved 4031 rows to data/GC=F.csv
[27/31] Downloading SI=F...
    local data already covers requested window; no download needed
    saved 4031 rows to data/SI=F.csv
[28/31] Downloading CL=F...
    local data already covers requested window; no download needed
    saved 4032 rows to data/CL=F.csv
[29/31] Downloading CSH.PA...
    local data already covers requested window; no download needed
    saved 4099 rows to data/CSH.PA.csv
[30/31] Downloading PDBC...
    local data already covers requested window; no download needed
    saved 2873 rows to data/PDBC.csv
[31/31] Downloading VGLT...


/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:322: RuntimeWarning: Local cache miss for symbols: ['VGLT']. Attempting online providers.
  f"Local cache miss for symbols: {missing_only}. Attempting online providers.",
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/yfinance/scrapers/history.py:169: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


    saved 4032 rows to data/VGLT.csv


,ticker,status,rows,error
0,CSSPX.MI,ok,4038,
1,SWDA.MI,ok,4069,
2,EIMI.MI,ok,2935,
3,CSNDX.MI,ok,4063,
4,IUSQ.MI,failed,0,No price data returned by yfinance for request...
5,VWCE.MI,failed,0,No price data returned by yfinance for request...
6,EXS1.MI,ok,4062,
7,MEUD.MI,failed,0,No price data returned by yfinance for request...
8,SMEA.MI,failed,0,No price data returned by yfinance for request...
9,XD9U.MI,ok,2651,
